# Named Entity Recognition

Ensure that FastAPI is up and running. [instructions here](../fast_api/readme.md)

# Table of contents

1. Data prep
2. Standard
        2.1. Train Model
        2.2. Predict Labels
        2.3. Evaluate Model
3. Trigger
        3.1. Soft Match
            3.1.1. Train Model
            3.1.2. Predict Labels
            3.1.3. Evaluate Model
<!--         3.2. Strict Match
            3.2.1. Train Model
            3.2.2. Predict Labels
            3.2.3. Evaluate Model -->

In [1]:
# imports
import requests
import json

In [2]:
FAST_API_URL = "http://localhost:9000"

 ---
 # 1. Data prep

We use a subset of [CONLL 2003 NER benchmark dataset](https://paperswithcode.com/sota/named-entity-recognition-ner-on-conll-2003) dataset in this demo.

---
#### Prepare data for Standard training

In [3]:
# load labelled data without explanations for Standard training
ner_standard_labeled_data = json.load(open('data/ner_conll/train_20.json'))
ner_standard_labeled_data[0]

{'text': 'EU rejects German call to boycott British lamb .',
 'label': 'B-ORG O B-MISC O O O B-MISC O O'}

In [4]:
# load dev data for Standard training
ner_standard_dev_data = json.load(open('data/ner_conll/dev.json'))
ner_standard_dev_data[0]

{'text': 'CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTER INNINGS VICTORY .',
 'label': 'O O B-ORG O O O O O O O O'}

---
#### Prepare data for Trigger training

In [5]:
# load labelled data with explanations for Trigger training
ner_trigger_labeled_data = json.load(open('data/ner_conll/trigger_20_explanation.json'))
ner_trigger_labeled_data[0]

{'text': 'EU rejects German call to boycott British lamb .',
 'label': 'B-ORG O O O O O O O O',
 'explanation': 'O T-3 T-0 T-4 O T-1 T-2 T-2 O'}

---
#### Prepare data for evaluation

In [6]:
# load test data for Standard training
ner_standard_test_data_dict = json.load(open('data/ner_conll/test.json'))
ner_standard_test_data = list(map(
    lambda x: [x['text'], x['label']],
    ner_standard_test_data_dict
))
ner_standard_test_data[0]

['SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .',
 'O O B-LOC O O O O B-PER O O O O']

---
#### Prepare data for prediction

In [7]:
ner_predict_data = list(map(lambda x: x[0], ner_standard_test_data))[:10]
ner_predict_data[0]

'SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .'

---
---
# 2. Standard

Check [this](../fast_api/json_schema.py#L516) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 2.1. Train model

### Define training parameters

In [8]:
ner_standard_train_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_standard",
    # a string name representing the dataset name
    "dataset_name": "conll",
    # task type - "ner" for Named Entity REcognition
    "task": "ner",
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True,
    "num_epochs": 10,
    # training batch size
    "batch_size": 10,
    # learning rate
    "learning_rate": 0.01,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
    # random seed
    "seed": 1337,
}

### Run model training

In [9]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/standard/ner/api/',
    json={
        'params': ner_standard_train_params,
        'labeled_data': ner_standard_labeled_data,
        'dev_data': ner_standard_dev_data,
        'eval_data': ner_standard_test_data_dict,
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/ink-lab-lean-life/LEAN-LIFE/model_api/fast_api/../model_training/trigger_ner/utilities/../../generated_data/saved_models/naive_conll_glove.6B.100d_1337_-1.0"}'

---
## 2.2. Predict labels

### Define prediction parameters

In [10]:
ner_standard_predict_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_standard",
    # a string name representing the dataset name
    "dataset_name": "conll",
    # task type - "ner" for Named Entity Recognition
    "task": "ner",
    # prediction batch size
    "batch_size": 10,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
}

### Fetch predictions

In [11]:
response = requests.post(
    FAST_API_URL + '/training/standard/ner/predict/',
    json={
        'params': ner_standard_predict_params,
        'prediction_data': ner_predict_data,
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list representing the label that was predicted
response.text

'{"class_preds":["O O S-LOC I-MISC E-MISC O O S-ORG O B-ORG E-ORG O","B-ORG I-ORG","S-ORG O B-ORG I-ORG E-ORG O","S-LOC O O O O O B-MISC E-MISC O O O O O O O S-LOC O O B-MISC E-MISC O O O O O","O S-LOC O O O O O O O O O O O O O O O O O O O O O S-LOC O","S-LOC O O O O O O O O O O O O O O O S-ORG O B-PER E-PER O O O O O O O O O O O O O O S-MISC O O O O O O O","B-PER E-PER O O O O O O O O O O O O O O O O O O O O O","O O S-MISC O O O O O B-MISC E-MISC O O O O O O O","O O O B-MISC E-MISC O O O O O S-LOC O O O O O O O","O O O O O O O O O O O S-LOC O O O O O O O O O O O O O O S-LOC O"]}'

---
## 2.3. Evaluate model

### Define evaluation parameters

In [12]:
ner_standard_eval_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_standard",
    # a string name representing the dataset name
    "dataset_name": "conll",
    # task type - "ner" for Named Entity Recognition
    "task": "ner",
    # evaluation batch size
    "batch_size": 10,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
}

### Run model evaluation

In [13]:
response = requests.post(
    FAST_API_URL + '/training/standard/ner/eval/',
    json={
        'params': ner_standard_eval_params,
        'eval_data': ner_standard_test_data,
    }
)
# returns JSON object with keys
# "precision", "recall" and "f1"
response.text

'{"precision":74.06635400382143,"recall":75.4957507082153,"f1":74.77422183252959}'

---
---
# 3. Trigger

Check [this](../fast_api/json_schema.py#L516) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

## 3.1. Train model

### Define training parameters

In [14]:
ner_trigger_train_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_soft_match",
    # a string name representing the dataset name
    "dataset_name": "conll_trigger",
    # task type - "ner" for Named Entity REcognition
    "task": "ner",
    # when "True" data has to be passed, 
    # "False" when re-training or the data was processed earlier and can be retrieved
    "build_data": True,
    # number of epochs
    "num_epochs": 10,
    # number of pre training epochs
    "pre_train_num_epochs": 20,
    # training batch size
    "batch_size": 10,
    # learning rate
    "learning_rate": 0.01,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
    # random seed
    "seed": 1337,
}

### Run model training

In [15]:
# depending on input size, and computing environment this might take time.
# please check FAST API logs for updates
response = requests.post(
    FAST_API_URL + '/training/trigger/api/',
    json={
        'params': ner_trigger_train_params,
        'explanation_triples': ner_trigger_labeled_data,
        'dev_data': ner_standard_dev_data,
        'eval_data': ner_standard_test_data_dict,
    }
)
# JSON with "save_path" key is returned when successful
response.text

'{"save_path":"/home/ink-lab-lean-life/LEAN-LIFE/model_api/fast_api/../model_training/trigger_ner/utilities/../../generated_data/saved_models/trigger_conll_trigger_glove.6B.100d_1337_-1.0"}'

---
## 3.2. Predict labels

### Define prediction parameters

In [16]:
ner_trigger_predict_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_soft_match",
    # a string name representing the dataset name
    "dataset_name": "conll_trigger",
    # task type - "ner" for Named Entity Recognition
    "task": "ner",
    # prediction batch size
    "batch_size": 10,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
}

### Fetch predictions

In [17]:
response = requests.post(
    FAST_API_URL + '/training/trigger/predict/',
    json={
        'params': ner_trigger_predict_params,
        'prediction_data': ner_predict_data,
    }
)
# returns JSON object with keys
# "class_probs" - list of list representing predictions for each label
# "class_preds" - list representing the label that was predicted
response.text

'{"class_preds":["O","O","O","O","O","O","O","O","O","O"]}'

---
## 3.3. Evaluate model

### Define evaluation parameters

In [18]:
ner_trigger_eval_params = {
    # a string name representing the model name
    "experiment_name": "conll_ner_soft_match",
    # a string name representing the dataset name
    "dataset_name": "conll_trigger",
    # task type - "ner" for Named Entity Recognition
    "task": "re",
    # evaluation batch size
    "batch_size": 10,
    # embedding to be used for training. usual default: "glove.6B.100d"
    "embeddings": "glove.6B.100d",
    # embedding dimension of the "embeddings" provided
    "emb_dim": 100,
    # number of hidden dimensions
    "hidden_dim": 200,
}

### Run model evaluation

In [19]:
response = requests.post(
    FAST_API_URL + '/training/trigger/eval/',
    json={
        'params': ner_trigger_eval_params,
        'eval_data': ner_standard_test_data,
    }
)
# returns JSON object with keys
# "precision", "recall" and "f1"
response.text

'{"precision":77.3704617553754,"recall":77.72662889518413,"f1":77.54813637166578}'